<a href="https://colab.research.google.com/github/onoway68/DA-Tech-Health-AI-Platform/blob/main/Copy_of_clinical_llm_qlora_v0_2_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Clinical Reasoning QLoRA — v0.2.1

Base model: `Qwen/Qwen3-4B` @ `1cfa9a7208912126459214e8b04321603b3df60c`

Training input:
- train: 60 records — `324062a9c5d2622dcc5b6c32465ca9e19d83060a7381f0a194273b3b1801b303`
- validation: 20 records — `d5f0ff02c6dd169228a82c8fc56810b80fd7ccaedaf7ab6c7baa1da598399310`

Forbidden frozen test coordinate:
`3900545900d95e7b901feecc5904e1e45ee57334cf6789ffd72cbfce15a21924`

**The training ZIP contains no test split.** This notebook fails closed on dataset hashes and model revision.


## 1 — GPU gate

In [ ]:
import torch, platform
assert torch.cuda.is_available(), "FAIL: GPU required"
print("GPU:",torch.cuda.get_device_name(0)); print("Python:",platform.python_version())


GPU: Tesla T4
Python: 3.13.15


## 2 — Install QLoRA stack

In [ ]:
!pip -q install "transformers>=4.52.4" "peft>=0.17" "trl>=0.20" "bitsandbytes>=0.46" accelerate datasets safetensors sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 13.4 MB/s eta 0:00:00


## 3 — Upload training-only ZIP

In [ ]:
from google.colab import files
uploaded=files.upload()
NAME="clinical-llm-training-only-v0.2.1.zip"
assert NAME in uploaded, f"FAIL: expected {NAME}"


Saving clinical-llm-training-only-v0.2.1.zip to clinical-llm-training-only-v0.2.1.zip


## 4 — Verify 60/20 training coordinates and absence of test data

In [ ]:
import pathlib,zipfile,shutil,json,hashlib
ROOT=pathlib.Path("/content/clinical_qlora")
if ROOT.exists(): shutil.rmtree(ROOT)
ROOT.mkdir()
with zipfile.ZipFile("/content/clinical-llm-training-only-v0.2.1.zip") as z:z.extractall(ROOT/"input")
def sha256(p):return hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest()
TRAIN=ROOT/"input/train.jsonl"; VAL=ROOT/"input/validation.jsonl"; MAN=ROOT/"input/training_input_manifest.json"
assert not (ROOT/"input/test.jsonl").exists(),"FAIL: test data present in training package"
assert sha256(TRAIN)=="324062a9c5d2622dcc5b6c32465ca9e19d83060a7381f0a194273b3b1801b303","FAIL: train SHA mismatch"
assert sha256(VAL)=="d5f0ff02c6dd169228a82c8fc56810b80fd7ccaedaf7ab6c7baa1da598399310","FAIL: validation SHA mismatch"
manifest=json.loads(MAN.read_text())
assert manifest["test_data_included"] is False
assert manifest["forbidden_test_sha256"]=="3900545900d95e7b901feecc5904e1e45ee57334cf6789ffd72cbfce15a21924"
train_rows=[json.loads(x) for x in TRAIN.read_text().splitlines() if x.strip()]
val_rows=[json.loads(x) for x in VAL.read_text().splitlines() if x.strip()]
assert len(train_rows)==60 and len(val_rows)==20
print("PASS — training-only boundary verified")
print("train:",len(train_rows),sha256(TRAIN))
print("validation:",len(val_rows),sha256(VAL))
print("test present: False")


PASS — training-only boundary verified
train: 60 324062a9c5d2622dcc5b6c32465ca9e19d83060a7381f0a194273b3b1801b303
validation: 20 d5f0ff02c6dd169228a82c8fc56810b80fd7ccaedaf7ab6c7baa1da598399310
test present: False


## 5 — Load exact base revision in 4-bit NF4

In [ ]:
import transformers, torch
from transformers import AutoTokenizer,AutoModelForCausalLM,BitsAndBytesConfig
MODEL_ID="Qwen/Qwen3-4B"; REV="1cfa9a7208912126459214e8b04321603b3df60c"
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID,revision=REV)
bnb=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type="nf4",bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=torch.float16)
model=AutoModelForCausalLM.from_pretrained(MODEL_ID,revision=REV,quantization_config=bnb,device_map="auto")
assert getattr(model.config,"_commit_hash",None)==REV,"FAIL: base revision mismatch"
model.config.use_cache=False
print("PASS — exact base revision loaded:",REV)


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (2 outputs left out)

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

PASS — exact base revision loaded: 1cfa9a7208912126459214e8b04321603b3df60c


## 6 — Build assistant-only supervised examples

In [ ]:
from datasets import Dataset
def encode_record(r):
    # Metadata is deliberately excluded. Full conversation is formatted only to
    # identify assistant-token positions; labels before assistant response are -100.
    sysu=[r["messages"][0],r["messages"][1]]
    full=[r["messages"][0],r["messages"][1],r["messages"][2]]
    prefix=tokenizer.apply_chat_template(sysu,tokenize=False,add_generation_prompt=True,enable_thinking=False)
    fulltext=tokenizer.apply_chat_template(full,tokenize=False,add_generation_prompt=False,enable_thinking=False)
    prefix_ids=tokenizer(prefix,add_special_tokens=False)["input_ids"]
    enc=tokenizer(fulltext,add_special_tokens=False,truncation=True,max_length=2048)
    labels=enc["input_ids"].copy()
    cutoff=min(len(prefix_ids),len(labels))
    labels[:cutoff]=[-100]*cutoff
    assert any(x!=-100 for x in labels),"FAIL: no assistant target tokens"
    return {"input_ids":enc["input_ids"],"attention_mask":enc["attention_mask"],"labels":labels}
train_ds=Dataset.from_list([encode_record(r) for r in train_rows])
val_ds=Dataset.from_list([encode_record(r) for r in val_rows])
print("PASS — assistant-only labels:",len(train_ds),len(val_ds))


PASS — assistant-only labels: 60 20


## 7 — Configure LoRA

In [ ]:
from peft import LoraConfig,prepare_model_for_kbit_training,get_peft_model
model=prepare_model_for_kbit_training(model)
lora=LoraConfig(r=16,lora_alpha=32,lora_dropout=0.05,bias="none",task_type="CAUSAL_LM",
 target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])
model=get_peft_model(model,lora)
model.print_trainable_parameters()


trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145


## 8 — Train with validation only

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq, set_seed

set_seed(20260907)
OUT = ROOT / "adapter"

args = TrainingArguments(
    output_dir=str(OUT),
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_steps=0.05,
    logging_steps=1,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    report_to="none",
    seed=20260907,
    data_seed=20260907,
)

collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    label_pad_token_id=-100,
    return_tensors="pt",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
)

result = trainer.train()
print(result)

Epoch,Training Loss,Validation Loss
1,1.608171,1.686700
2,1.322708,1.444033
3,1.146702,1.397861


TrainOutput(global_step=24, training_loss=1.6095445305109024, metrics={'train_runtime': 372.4134, 'train_samples_per_second': 0.483, 'train_steps_per_second': 0.064, 'total_flos': 1752702444334080.0, 'train_loss': 1.6095445305109024, 'epoch': 3.0})


## 9 — Save adapter and evidence manifest

In [ ]:
import os,datetime,hashlib,json
FINAL=ROOT/"final_adapter"; model.save_pretrained(FINAL); tokenizer.save_pretrained(FINAL)
def tree_hash(d):
    h=hashlib.sha256()
    for p in sorted(pathlib.Path(d).rglob("*")):
        if p.is_file():
            h.update(str(p.relative_to(d)).encode());h.update(p.read_bytes())
    return h.hexdigest()
evidence={"experiment_version":"0.2.1","base_model":MODEL_ID,"base_revision":REV,
"train_sha256":"324062a9c5d2622dcc5b6c32465ca9e19d83060a7381f0a194273b3b1801b303","validation_sha256":"d5f0ff02c6dd169228a82c8fc56810b80fd7ccaedaf7ab6c7baa1da598399310","forbidden_test_sha256":"3900545900d95e7b901feecc5904e1e45ee57334cf6789ffd72cbfce15a21924",
"test_used_for_training":False,"method":"QLoRA","quantization":"4-bit NF4 double quantization",
"lora":{"r":16,"alpha":32,"dropout":0.05,"targets":["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]},
"training":{"epochs":3,"batch_size":1,"gradient_accumulation":8,"learning_rate":1e-4,"max_length":2048,"seed":20260907},
"runtime":{"python":platform.python_version(),"torch":torch.__version__,"transformers":transformers.__version__,"gpu":torch.cuda.get_device_name(0)},
"adapter_tree_sha256":tree_hash(FINAL),"completed_utc":datetime.datetime.now(datetime.timezone.utc).isoformat()}
(ROOT/"training_manifest.json").write_text(json.dumps(evidence,indent=2)+"\n")
print(json.dumps(evidence,indent=2))


{
  "experiment_version": "0.2.1",
  "base_model": "Qwen/Qwen3-4B",
  "base_revision": "1cfa9a7208912126459214e8b04321603b3df60c",
  "train_sha256": "324062a9c5d2622dcc5b6c32465ca9e19d83060a7381f0a194273b3b1801b303",
  "validation_sha256": "d5f0ff02c6dd169228a82c8fc56810b80fd7ccaedaf7ab6c7baa1da598399310",
  "forbidden_test_sha256": "3900545900d95e7b901feecc5904e1e45ee57334cf6789ffd72cbfce15a21924",
  "test_used_for_training": false,
  "method": "QLoRA",
  "quantization": "4-bit NF4 double quantization",
  "lora": {
    "r": 16,
    "alpha": 32,
    "dropout": 0.05,
    "targets": [
      "q_proj",
      "k_proj",
      "v_proj",
      "o_proj",
      "gate_proj",
      "up_proj",
      "down_proj"
    ]
  },
  "training": {
    "epochs": 3,
    "batch_size": 1,
    "gradient_accumulation": 8,
    "learning_rate": 0.0001,
    "max_length": 2048,
    "seed": 20260907
  },
  "runtime": {
    "python": "3.13.15",
    "torch": "2.11.0+cu128",
    "transformers": "5.16.1",
    "gpu": "Tesla

## 10 — Export training evidence

In [ ]:
bundle=shutil.make_archive("/content/clinical-llm-qlora-training-evidence-v0.2.1","zip",ROOT)
files.download(bundle)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pathlib
import zipfile
import json
import hashlib

ROOT = pathlib.Path("/content/clinical_qlora")
FINAL = ROOT / "final_adapter"
MANIFEST = ROOT / "training_manifest.json"

assert FINAL.exists(), "FAIL: final_adapter directory missing"
assert MANIFEST.exists(), "FAIL: training_manifest.json missing"

manifest = json.loads(MANIFEST.read_text())

EXPECTED_ADAPTER_SHA = (
    "6280a4cbd8ea94cc4bbe470f460384cc5639d13d4c135f1da9b73b1764e7b46e"
)

def tree_hash(directory):
    h = hashlib.sha256()

    for p in sorted(pathlib.Path(directory).rglob("*")):
        if p.is_file():
            h.update(str(p.relative_to(directory)).encode())
            h.update(p.read_bytes())

    return h.hexdigest()

actual_adapter_sha = tree_hash(FINAL)

assert actual_adapter_sha == EXPECTED_ADAPTER_SHA, (
    f"FAIL: adapter hash mismatch: {actual_adapter_sha}"
)

assert manifest["adapter_tree_sha256"] == EXPECTED_ADAPTER_SHA
assert manifest["test_used_for_training"] is False

OUT = pathlib.Path(
    "/content/clinical-llm-qlora-final-adapter-v0.2.1.zip"
)

with zipfile.ZipFile(
    OUT,
    "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=9,
) as z:

    # Final trained adapter + tokenizer/config files
    for p in sorted(FINAL.rglob("*")):
        if p.is_file():
            z.write(
                p,
                pathlib.Path("final_adapter") / p.relative_to(FINAL)
            )

    # Experiment manifest
    z.write(
        MANIFEST,
        "training_manifest.json"
    )

print("PASS — compact adapter evidence created")
print("Adapter SHA-256:", actual_adapter_sha)
print("Output:", OUT)
print(
    "ZIP size:",
    round(OUT.stat().st_size / (1024 * 1024), 2),
    "MB"
)

PASS — compact adapter evidence created
Adapter SHA-256: 6280a4cbd8ea94cc4bbe470f460384cc5639d13d4c135f1da9b73b1764e7b46e
Output: /content/clinical-llm-qlora-final-adapter-v0.2.1.zip
ZIP size: 118.39 MB


In [ ]:
from google.colab import files
files.download(
    "/content/clinical-llm-qlora-final-adapter-v0.2.1.zip"
)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import json
import pathlib
import torch
import random
import numpy as np
from transformers import set_seed

# -----------------------------
# Validation inference gate
# -----------------------------

EXPECTED_VAL_SHA = (
    "d5f0ff02c6dd169228a82c8fc56810b80fd7ccaedaf7ab6c7baa1da598399310"
)

EXPECTED_ADAPTER_SHA = (
    "6280a4cbd8ea94cc4bbe470f460384cc5639d13d4c135f1da9b73b1764e7b46e"
)

assert sha256(VAL) == EXPECTED_VAL_SHA, "FAIL: validation SHA mismatch"

assert not (ROOT / "input/test.jsonl").exists(), (
    "FAIL: frozen test data unexpectedly present"
)

assert manifest["adapter_tree_sha256"] == EXPECTED_ADAPTER_SHA, (
    "FAIL: wrong trained adapter"
)

# Same generation coordinates as baseline
GENERATION = {
    "enable_thinking": False,
    "do_sample": True,
    "temperature": 0.7,
    "top_p": 0.8,
    "top_k": 20,
    "max_new_tokens": 1024,
    "seed": 20260907,
}

set_seed(GENERATION["seed"])
random.seed(GENERATION["seed"])
np.random.seed(GENERATION["seed"])

model.eval()

validation_predictions = []

for i, record in enumerate(val_rows, start=1):

    # IMPORTANT:
    # Only system + user are supplied to the model.
    # Gold assistant response is NOT included.
    inference_messages = [
        record["messages"][0],
        record["messages"][1],
    ]

    prompt = tokenizer.apply_chat_template(
        inference_messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False,
    )

    inputs = {
        k: v.to(model.device)
        for k, v in inputs.items()
    }

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            do_sample=True,
            temperature=0.7,
            top_p=0.8,
            top_k=20,
            max_new_tokens=1024,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = output[0][inputs["input_ids"].shape[1]:]

    response = tokenizer.decode(
        generated,
        skip_special_tokens=True,
    ).strip()

    validation_predictions.append({
        "id": record["id"],
        "prediction": response,
    })

    print(
        f"[{i:02d}/{len(val_rows):02d}] "
        f"{record['id']} — {len(generated)} tokens"
    )

VAL_PRED = ROOT / "validation_predictions.json"

VAL_PRED.write_text(
    json.dumps(
        validation_predictions,
        indent=2,
        ensure_ascii=False,
    ) + "\n"
)

print()
print("PASS — validation inference completed")
print("Records:", len(validation_predictions))
print("Predictions SHA-256:", sha256(VAL_PRED))
print("Frozen test present:", False)

[01/20] CR-008 — 244 tokens
[02/20] CR-010 — 352 tokens
[03/20] CR-017 — 286 tokens
[04/20] CR-025 — 290 tokens
[05/20] CR-026 — 271 tokens
[06/20] CR-028 — 226 tokens
[07/20] CR-032 — 307 tokens
[08/20] CR-035 — 244 tokens
[09/20] CR-040 — 180 tokens
[10/20] CR-049 — 1024 tokens
[11/20] CR-053 — 269 tokens
[12/20] CR-057 — 362 tokens
[13/20] CR-066 — 1024 tokens
[14/20] CR-067 — 239 tokens
[15/20] CR-070 — 241 tokens
[16/20] CR-071 — 1024 tokens
[17/20] CR-073 — 289 tokens
[18/20] CR-076 — 277 tokens
[19/20] CR-086 — 1024 tokens
[20/20] CR-088 — 282 tokens

PASS — validation inference completed
Records: 20
Predictions SHA-256: ed66314c205cd41b0baf9a973183bbd6404a9dc02781e6294dcd30c4f67c3d4f
Frozen test present: False


In [ ]:
import json
import re
import hashlib

EXPECTED_PRED_SHA = (
    "ed66314c205cd41b0baf9a973183bbd6404a9dc02781e6294dcd30c4f67c3d4f"
)

assert sha256(VAL_PRED) == EXPECTED_PRED_SHA, (
    "FAIL: validation predictions changed"
)

REQUIRED_HEADERS = [
    "PROBLEM REPRESENTATION:",
    "RELEVANT FINDINGS:",
    "POSSIBLE EXPLANATIONS:",
    "MISSING INFORMATION:",
    "RED FLAGS:",
    "UNCERTAINTY:",
]

def evaluate_schema(text):
    positions = []
    counts = {}

    for header in REQUIRED_HEADERS:
        count = text.count(header)
        counts[header] = count

        if count == 1:
            positions.append(text.find(header))
        else:
            positions.append(-1)

    exactly_once = all(
        counts[h] == 1
        for h in REQUIRED_HEADERS
    )

    correct_order = (
        exactly_once
        and positions == sorted(positions)
    )

    # No text before first required heading.
    clean_start = text.startswith(
        REQUIRED_HEADERS[0]
    )

    exact_schema = (
        exactly_once
        and correct_order
        and clean_start
    )

    return {
        "exact_schema": exact_schema,
        "exactly_once": exactly_once,
        "correct_order": correct_order,
        "clean_start": clean_start,
        "header_counts": counts,
    }


schema_results = []

for row in validation_predictions:

    result = evaluate_schema(
        row["prediction"]
    )

    schema_results.append({
        "id": row["id"],
        **result,
    })


passed = sum(
    r["exact_schema"]
    for r in schema_results
)

total = len(schema_results)

SCHEMA_REPORT = ROOT / "validation_schema_report.json"

SCHEMA_REPORT.write_text(
    json.dumps(
        schema_results,
        indent=2,
        ensure_ascii=False,
    ) + "\n"
)

print("VALIDATION STRUCTURAL GATE")
print("--------------------------")

for r in schema_results:
    status = (
        "PASS"
        if r["exact_schema"]
        else "FAIL"
    )

    print(
        f"{r['id']}: {status}"
    )

print()
print(
    f"Exact schema compliance: "
    f"{passed}/{total} "
    f"({100 * passed / total:.1f}%)"
)

print(
    "Schema report SHA-256:",
    sha256(SCHEMA_REPORT)
)

print(
    "Predictions SHA-256:",
    sha256(VAL_PRED)
)

print(
    "Frozen test present:",
    False
)


VALIDATION STRUCTURAL GATE
--------------------------
CR-008: PASS
CR-010: FAIL
CR-017: PASS
CR-025: FAIL
CR-026: PASS
CR-028: FAIL
CR-032: PASS
CR-035: PASS
CR-040: FAIL
CR-049: FAIL
CR-053: PASS
CR-057: PASS
CR-066: FAIL
CR-067: PASS
CR-070: FAIL
CR-071: FAIL
CR-073: FAIL
CR-076: FAIL
CR-086: FAIL
CR-088: PASS

Exact schema compliance: 9/20 (45.0%)
Schema report SHA-256: b80eb989d4c7ae8b08f63a14455e4e1afebda86bdf48944b8aac387cf39b551f
Predictions SHA-256: ed66314c205cd41b0baf9a973183bbd6404a9dc02781e6294dcd30c4f67c3d4f
Frozen test present: False


In [ ]:
import pathlib
import hashlib
import json
import re
from collections import Counter

# -----------------------------------
# Self-contained integrity functions
# -----------------------------------

def sha256(path):
    path = pathlib.Path(path)

    h = hashlib.sha256()

    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)

    return h.hexdigest()


EXPECTED_PRED_SHA = (
    "ed66314c205cd41b0baf9a973183bbd6404a9dc02781e6294dcd30c4f67c3d4f"
)

EXPECTED_SCHEMA_SHA = (
    "b80eb989d4c7ae8b08f63a14455e4e1afebda86bdf48944b8aac387cf39b551f"
)

# -----------------------------------
# Recover paths if variables vanished
# -----------------------------------

ROOT = pathlib.Path("/content/clinical_qlora")

VAL_PRED = ROOT / "validation_predictions.json"
SCHEMA_REPORT = ROOT / "validation_schema_report.json"

assert VAL_PRED.exists(), (
    "FAIL: validation_predictions.json missing"
)

assert SCHEMA_REPORT.exists(), (
    "FAIL: validation_schema_report.json missing"
)

assert sha256(VAL_PRED) == EXPECTED_PRED_SHA, (
    "FAIL: validation predictions changed"
)

assert sha256(SCHEMA_REPORT) == EXPECTED_SCHEMA_SHA, (
    "FAIL: schema report changed"
)

assert not (ROOT / "input/test.jsonl").exists(), (
    "FAIL: frozen test unexpectedly present"
)

# -----------------------------------
# Reload locked evidence from disk
# -----------------------------------

validation_predictions = json.loads(
    VAL_PRED.read_text()
)

schema_results = json.loads(
    SCHEMA_REPORT.read_text()
)

assert len(validation_predictions) == 20
assert len(schema_results) == 20

prediction_by_id = {
    x["id"]: x["prediction"]
    for x in validation_predictions
}

schema_by_id = {
    x["id"]: x
    for x in schema_results
}


# -----------------------------------
# Structural failure diagnostic
# -----------------------------------

print("STRUCTURAL FAILURE DIAGNOSTIC")
print("-----------------------------")

failed_ids = [
    r["id"]
    for r in schema_results
    if not r["exact_schema"]
]

for case_id in failed_ids:

    r = schema_by_id[case_id]
    text = prediction_by_id[case_id]

    missing = [
        h
        for h, count in r["header_counts"].items()
        if count == 0
    ]

    duplicated = [
        h
        for h, count in r["header_counts"].items()
        if count > 1
    ]

    print()
    print(case_id)
    print("  clean_start:", r["clean_start"])
    print("  exactly_once:", r["exactly_once"])
    print("  correct_order:", r["correct_order"])
    print("  missing:", missing or "NONE")
    print("  duplicated:", duplicated or "NONE")

    section_lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip().endswith(":")
    ]

    print(
        "  observed section lines:",
        section_lines[:20] or "NONE"
    )


# -----------------------------------
# Max-token diagnostic
# -----------------------------------

print()
print("MAX-TOKEN CASE DIAGNOSTIC")
print("-------------------------")

MAX_TOKEN_IDS = [
    "CR-049",
    "CR-066",
    "CR-071",
    "CR-086",
]

for case_id in MAX_TOKEN_IDS:

    text = prediction_by_id[case_id]

    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

    normalized = [
        re.sub(r"\s+", " ", line).lower()
        for line in lines
    ]

    counts = Counter(normalized)

    repeated = [
        (line, count)
        for line, count in counts.items()
        if count >= 3
    ]

    repeated.sort(
        key=lambda x: (-x[1], x[0])
    )

    print()
    print(case_id)
    print("  characters:", len(text))
    print("  nonempty lines:", len(lines))

    print(
        "  repeated lines >=3:",
        repeated[:10] or "NONE"
    )

    print("  final 500 characters:")
    print("  ---")
    print(text[-500:])
    print("  ---")


# -----------------------------------
# Final integrity confirmation
# -----------------------------------

print()
print("INTEGRITY CONFIRMATION")
print("----------------------")

print(
    "Predictions SHA-256:",
    sha256(VAL_PRED)
)

print(
    "Schema report SHA-256:",
    sha256(SCHEMA_REPORT)
)

print(
    "Frozen test present:",
    False
)

AssertionError: FAIL: validation_predictions.json missing

In [ ]:
import pathlib

print("SEARCHING FOR LOCKED VALIDATION EVIDENCE")
print("----------------------------------------")

names = [
    "validation_predictions.json",
    "validation_schema_report.json",
]

found = []

for name in names:
    for path in pathlib.Path("/content").rglob(name):
        if path.is_file():
            found.append(path)
            print(path)

print()
print("Files found:", len(found))

SEARCHING FOR LOCKED VALIDATION EVIDENCE
----------------------------------------

Files found: 0
